In [2]:
# Install dependencies
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.1 MB/s eta 0:00:00


In [12]:
# Load env variables
from anthropic import Anthropic
from google.colab import userdata
from IPython.display import Markdown
from anthropic.types import ToolParam
from datetime import datetime, timedelta
from anthropic.types import Message
import json


In [13]:
# 1. Retrieve the key from Colab Secrets
my_anthropic_key   = userdata.get('ANTHROPIC_API_KEY')
my_anthropic_model = userdata.get('CLAUDE_MODEL')


In [14]:

# Create an API client
client = Anthropic(
    api_key=my_anthropic_key,
)

print("Anthropic client initialized successfully!")


Anthropic client initialized successfully!


In [15]:
# define a system prompt
#system_prompt = """
#You are a creative screenwriter for feature films.
#"""
system_prompt = None

In [24]:
# Utility functions
def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": my_anthropic_model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)

def chat(messages, system=None, stop_sequences=[], tools=None):
    params = {
        "model": my_anthropic_model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)

    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [17]:
# Tools and Schemas
save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)

save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)

def save_article(**kwargs):
    return "Article saved!"


In [18]:
def get_tool_use_block(response):
    """
    Safely searches through Claude's response and returns the first ToolUseBlock.
    Returns None if no tool was called.
    """
    for block in response.content:
        if block.type == 'tool_use':
            return block

    return None

def get_tool_use_blocks(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    return tool_requests


In [19]:
def assemble_tool_result(tool_request, tool_output):
  tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_request.id,
    "content": json.dumps(tool_output),
    "is_error": False
  }
  return tool_result_block

def assemble_error_tool_result(tool_request, exception):
  tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_request.id,
    "content": f"Error: {exception}",
    "is_error": True
  }
  return tool_result_block


In [22]:
def run_tool(tool_request):
    tool_output = None
    tool_result_block = None
    tool_name = tool_request.name
    tool_input = tool_request.input
    if tool_name == "save_article":
        tool_output = save_article(**tool_input)
        tool_result_block = assemble_tool_result(tool_request, tool_output)
    return tool_result_block


In [21]:
def run_tools(message):
    tool_requests = get_tool_use_blocks(message)
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
          tool_result_block = run_tool(tool_request)
          tool_result_blocks.append(tool_result_block)
        except Exception as e:
          tool_result_block = assemble_error_tool_result(tool_request, e)
          tool_result_blocks.append(tool_result_block)

    return tool_result_blocks


In [10]:
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [26]:
messages = []

add_user_message(messages, "Create and save a fake computer science article")
run_conversation(messages, tools = [save_article_schema], fine_grained=True)



I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "A novel quantum-inspired algorithm for optimizing blockchain consensus mechanisms through recursive neural network pruning.", "meta": {
  "word_count": 8500,
  "review": "This paper presents an innovative approach to improving blockchain efficiency by combining quantum computing principles with deep learning optimization techniques. The authors introduce a recursive neural network pruning algorithm that demonstrates significant improvements in consensus validation times. The experimental evaluation on simulated blockchain networks shows promising results, though the real-world applicability remains unclear. The mathematical framework is rigorous and well-presented, with clear proofs of convergence properties. However, the paper lacks comprehensive comparison with existing state-of-the-art methods and relies heavily on synthetic benchmarks. The writing quality is generally good, t

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_017C8qX4HTndN6CZJEkei8UJ',
    'name': 'save_article',
    'input': {'abstract': 'A novel quantum-inspired algorithm for optimizing blockchain consensus mechanisms through recursive neural network pruning.',
     'meta': {'word_count': 8500,
      'review': 'This paper presents an innovative approach to improving blockchain efficiency by combining quantum computing principles with deep learning optimization techniques. The authors introduce a recursive neural network pruning algorithm that demonstrates significant improvements in consensus validation times. The experimental evaluation on simulated blockchain networks shows promising results, though the real-world applicability remains unclear. The mathe